# Track D / Day 5 — Trim to 400 + re-validate (Colab)

Implements `pilot_0_1_execution_spec.md` §2.2 step 5: trim the 420 Day-3 survivors down to the best 400 by dropping the worst length/perplexity outliers, then re-run Day 4's full battery on the result.

**Trim rule (`ghosts/DECISIONS.md` item 6, pre-registered before this ran):** 21 authors × 20 rows = 420; drop exactly **one whole author** — the one with the highest combined z-score of (mean length, mean perplexity) vs. holdout10 — leaving 20 × 20 = 400 rows, every surviving ghost a complete author.

**Heads up, stated in advance:** Day 4's real run failed all 3 tests, and the perplexity gap (d≈1.4–1.7) was present in both generators, not concentrated in a few rows. Dropping one author is the pre-registered next step regardless of outcome — it is **not** expected to fully flip FAIL to PASS on its own. This run tells us exactly how much it helps.

**Before running:** same as Day 4 — GPU runtime (Runtime → Change runtime type → T4 GPU) and an `HF_TOKEN` Colab secret. This script re-runs the same perplexity/SBERT computations Day 4 did (self-contained by design — see the script's module docstring), so it takes a similar amount of time.

## 1. Mount Drive and pull the repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/unlearning_pilot'
REPO_DIR = os.path.join(PROJECT_DIR, 'unlearning-audit-study')
os.makedirs(PROJECT_DIR, exist_ok=True)

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/shravanidhus31/unlearning-audit-study.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
!git log -1 --oneline

## 2. Confirm required files are present

In [ ]:
import os
for p in ['scripts/day5_trim.py', 'scripts/day4_validation.py', 'ghosts/candidates_filtered.jsonl']:
    assert os.path.exists(p), f'{p} not found -- re-run cell 1, or check Day 3/4 have been committed.'
print('All required files present.')

## 3. Install dependencies

In [ ]:
!pip install -q transformers datasets scipy sentence-transformers huggingface_hub torch

## 4. Secrets

In [ ]:
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
print('HF_TOKEN loaded:', bool(os.environ.get('HF_TOKEN')))

## 5. Self-test (offline — no network/GPU)

In [ ]:
!python scripts/day5_trim.py --selftest

## 6. Full run
Scores all 21 surviving authors, drops the worst one, writes `ghosts/final_400.jsonl`, then re-runs the full Day 4 battery on the result.

In [ ]:
!python scripts/day5_trim.py --outdir ghosts

## 7. Read the results

In [ ]:
with open('ghosts/trim_report.md', encoding='utf-8') as f:
    print(f.read())

## 8. Commit manually

In [ ]:
print('To commit manually:')
print('  git -C', REPO_DIR, 'add ghosts/final_400.jsonl ghosts/trim_report.md')
print('  git -C', REPO_DIR, 'commit -m "Track D Day 5: trim to 400 + re-validate"')
print('  git -C', REPO_DIR, 'push')